# Object Detection — YOLO, IoU, Anchor Boxes, NMS

> Based on Stanford CS230 — C4M3, Andrew Ng / DeepLearning.AI

Object detection outputs both a class label and a bounding box for every object in an image — simultaneously. YOLO (You Only Look Once) does this in a single forward pass, making it real-time capable.

## Learning Objectives

1. Define and compute Intersection over Union (IoU)
2. Explain anchor boxes and how they handle multiple objects per cell
3. Implement Non-Maximum Suppression (NMS) to remove duplicate detections
4. Understand the YOLO output grid and prediction format

## YOLO Output Format

Divide the image into an $S \times S$ grid. Each cell predicts $B$ bounding boxes, each described by:

$$[p_c,\; b_x,\; b_y,\; b_h,\; b_w,\; c_1,\; c_2,\; \ldots,\; c_C]$$

- $p_c$ — confidence (is there an object centred here?)
- $(b_x, b_y, b_h, b_w)$ — bounding box relative to grid cell
- $c_1 \ldots c_C$ — class probabilities (conditioned on object present)

Output volume: $S \times S \times B \times (5 + C)$

## Intersection over Union (IoU)

$$\text{IoU} = \frac{\text{Area of Intersection}}{\text{Area of Union}}$$

IoU $\geq 0.5$ is the standard threshold for "correct" detection.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# Bounding box: [x1, y1, x2, y2]
def iou(box_a, box_b):
    x1 = max(box_a[0], box_b[0]); y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2]); y2 = min(box_a[3], box_b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area_a = (box_a[2]-box_a[0]) * (box_a[3]-box_a[1])
    area_b = (box_b[2]-box_b[0]) * (box_b[3]-box_b[1])
    return inter / (area_a + area_b - inter + 1e-10)

def nms(boxes, scores, iou_thresh=0.5):
    """Non-Maximum Suppression."""
    order = np.argsort(scores)[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        ious = np.array([iou(boxes[i], boxes[j]) for j in order[1:]])
        order = order[1:][ious < iou_thresh]
    return keep

# ---- IoU visualisation ----
gt_box   = [1.0, 1.0, 4.0, 4.0]  # ground truth
pred_box = [2.0, 2.0, 5.0, 5.0]  # prediction
score = iou(gt_box, pred_box)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('IoU, YOLO Grid, and Non-Maximum Suppression', fontsize=13, fontweight='bold')

ax = axes[0]
ax.set_xlim(0, 6); ax.set_ylim(0, 6); ax.set_aspect('equal')
ax.set_title(f'Intersection over Union  =  {score:.2f}')
# Intersection box
ix1, iy1 = max(gt_box[0],pred_box[0]), max(gt_box[1],pred_box[1])
ix2, iy2 = min(gt_box[2],pred_box[2]), min(gt_box[3],pred_box[3])
ax.add_patch(patches.Rectangle((ix1,iy1), ix2-ix1, iy2-iy1, color=P[2], alpha=0.5, label='Intersection'))
ax.add_patch(patches.Rectangle((gt_box[0],gt_box[1]), gt_box[2]-gt_box[0], gt_box[3]-gt_box[1],
                                 fill=False, edgecolor=P[3], lw=2.5, label='Ground truth'))
ax.add_patch(patches.Rectangle((pred_box[0],pred_box[1]), pred_box[2]-pred_box[0], pred_box[3]-pred_box[1],
                                 fill=False, edgecolor=P[0], lw=2.5, ls='--', label='Prediction'))
ax.legend(fontsize=9); ax.grid(True)
ax.set_xlabel('x'); ax.set_ylabel('y')

# ---- YOLO grid ----
ax = axes[1]
ax.set_xlim(0, 3); ax.set_ylim(0, 3); ax.set_aspect('equal')
ax.set_title('YOLO 3×3 Grid\n(each cell predicts bounding boxes)')
for i in range(4):
    ax.axhline(i, color='#c8ccd4', lw=1)
    ax.axvline(i, color='#c8ccd4', lw=1)
# Simulated object centred in cell (1,1) — columns from left, rows from bottom
ax.add_patch(patches.Rectangle((0.3, 0.3), 1.8, 1.0, fill=False, edgecolor=P[1], lw=2, label='Object'))
ax.plot(1.2, 0.8, '*', color=P[2], ms=14, zorder=5, label='Object centre → cell (0,0)')
ax.text(0.5, 2.5, '(0,2)', ha='center', va='center', fontsize=9, color='#888')
ax.text(1.5, 2.5, '(1,2)', ha='center', va='center', fontsize=9, color='#888')
ax.text(2.5, 2.5, '(2,2)', ha='center', va='center', fontsize=9, color='#888')
ax.add_patch(patches.Rectangle((0,0), 1, 1, fill=True, facecolor=P[0], alpha=0.15, edgecolor=P[0], lw=2))
ax.text(0.5, 0.5, 'responsible\ncell', ha='center', va='center', fontsize=8, color=P[0])
ax.legend(fontsize=8); ax.set_xlabel('x'); ax.set_ylabel('y')

# ---- NMS ----
ax = axes[2]
ax.set_xlim(0,10); ax.set_ylim(0,10); ax.set_aspect('equal')
ax.set_title('Non-Maximum Suppression')
np.random.seed(3)
raw_boxes = [
    [1.0,1.0,5.0,5.0], [1.5,1.2,5.2,5.3], [1.8,0.8,4.8,4.8],  # cluster 1
    [6.0,6.0,9.0,9.0], [6.2,6.3,9.2,9.2],                       # cluster 2
]
raw_scores = [0.9, 0.75, 0.6, 0.85, 0.7]
keep_idx = nms(raw_boxes, raw_scores, iou_thresh=0.4)

for i, (box, score) in enumerate(zip(raw_boxes, raw_scores)):
    kept = i in keep_idx
    color = P[3] if kept else P[1]
    lw = 2.5 if kept else 1.2
    ls = '-' if kept else '--'
    ax.add_patch(patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                                    fill=kept, facecolor=color if kept else 'none',
                                    alpha=0.2 if kept else 1.0,
                                    edgecolor=color, lw=lw, ls=ls))
    ax.text((box[0]+box[2])/2, (box[1]+box[3])/2, f'{score:.2f}',
            ha='center', va='center', fontsize=9, color=color, fontweight='bold' if kept else 'normal')

ax.plot([], [], color=P[3], lw=2.5, label='Kept (highest IoU>0.4 suppressed)')
ax.plot([], [], color=P[1], lw=1.5, ls='--', label='Suppressed')
ax.legend(fontsize=8); ax.grid(True); ax.set_xlabel('x'); ax.set_ylabel('y')

plt.tight_layout()
plt.show()
print(f"Kept box indices: {keep_idx}  (scores: {[raw_scores[i] for i in keep_idx]})")
